In [1]:
%matplotlib inline
# Cell 1 — parameters
SPREAD_THRESHOLD        = 20    # percentile-point spread below which a variable is 'concentrated'
P_LOW                   = 10    # lower percentile for spread calculation
P_HIGH                  = 90    # upper percentile for spread calculation
ZERO_FRACTION_THRESHOLD = 0.20  # must match step 2; from work order 2026-06-14
ZERO_COVERAGE_THRESHOLD = 0.90  # buffer weight-at-zero fraction → 'outside_active_domain'

In [2]:
# Cell 2 — imports and load Step 2 outputs
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

sys.path.insert(0, '../../../..')
import scripts.shared.db_utils as _dbu

ROOT = Path(_dbu.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'edop' / 'areas'

# read_areas_tsv forces hybas_id and dominant_hybas_id to Int64 on every read
raw_df    = _dbu.read_areas_tsv(OUT / 'step2_raw.tsv',    index_col='hybas_id')
matrix_df = _dbu.read_areas_tsv(OUT / 'step2_matrix.tsv', index_col='hybas_id')
meta_df   = pd.read_csv(OUT / 'step2_meta.tsv', sep='\t', index_col='api_key')

print(f'raw_df    : {raw_df.shape}')
print(f'matrix_df : {matrix_df.shape}')
print(f'meta_df   : {meta_df.shape}')

raw_df    : (9, 55)
matrix_df : (9, 54)
meta_df   : (54, 8)


In [3]:
# Cell 3 — Step 3.1: join weights onto the matrix
#
# Both files use hybas_id as index; coerce to int before joining so float64
# representations (1060041510.0 vs 1060041510) don't cause mismatches.
# Inner join: any basin missing from either side is a mismatch — fail loudly.

weights = raw_df[['weight']].copy()
weights.index = weights.index.astype(int)

matrix = matrix_df.copy()
matrix.index = matrix.index.astype(int)

joined = weights.join(matrix, how='inner')

n_expected = len(weights)
n_joined   = len(joined)
weight_sum = joined['weight'].sum()

print(f'Basins in weights : {n_expected}')
print(f'Basins after join : {n_joined}  ({"OK" if n_joined == n_expected else "MISMATCH"})')
print(f'Weight sum        : {weight_sum:.6f}  ({"OK" if abs(weight_sum - 1.0) < 0.001 else "CHECK"})')

if n_joined != n_expected:
    missing = set(weights.index) - set(matrix.index)
    print(f'Missing hybas_ids: {missing}')

Basins in weights : 9
Basins after join : 9  (OK)
Weight sum        : 1.000000  (OK)


In [4]:
# Cell 4 — Step 3.2: select block-1 variables
#
# Block 1 handles two typology clusters via the same area-weighted coherence recipe:
#   continental-gradient — smooth spatially-autocorrelated fields (climate, soils, etc.)
#   scale-dependent      — field-like vars whose Moran's I varies across L6/L8
#                          (slope, karst, wetlands, cropland, groundwater, etc.)
#
# network-topology (discharge_annual, discharge_min, discharge_max) → Block 2.
# discharge_max was re-typed from scale-dependent → network-topology in catalog 2026-06-15;
# step2 must be re-run to propagate this to step2_meta.tsv and include it in Block 2.
#
# local-anomaly (river_area) → deferred to Block 5 fallback.

BLOCK1_CLUSTERS = {'continental-gradient', 'scale-dependent'}

block1_vars = meta_df[
    (meta_df['kind'] == 'continuous') &
    (meta_df['typology_cluster'].isin(BLOCK1_CLUSTERS))
].index.tolist()

block1_vars = [v for v in block1_vars if v in joined.columns]

print(f'Block 1 variables ({len(block1_vars)}):')
for v in block1_vars:
    row = meta_df.loc[v]
    print(f'  {v:35s}  cluster={row["typology_cluster"]}  band={row["band"]}  method={row["position_method"]}')

Block 1 variables (34):
  elev_min                             cluster=continental-gradient  band=A  method=percentile
  elev_max                             cluster=scale-dependent  band=A  method=percentile
  slope_avg                            cluster=scale-dependent  band=A  method=percentile
  slope_upstream                       cluster=scale-dependent  band=A  method=percentile
  stream_gradient                      cluster=scale-dependent  band=A  method=percentile
  karst                                cluster=scale-dependent  band=A  method=percentile
  karst_upstream                       cluster=scale-dependent  band=A  method=percentile
  permafrost_extent                    cluster=continental-gradient  band=C  method=percentile
  runoff                               cluster=continental-gradient  band=B  method=percentile
  gw_table_depth                       cluster=scale-dependent  band=B  method=percentile
  wet_pct_grp1                         cluster=scale-dependen

In [5]:
# Cell 5 — Step 3.3: weighted score distributions
#
# For each block-1 variable:
#   1. Pair scores with weights; drop basins where score is null.
#   2. Renormalize surviving weights to sum to 1.
#      (Null = data absence for that basin, not geographic absence;
#       renormalize rather than report a shortfall.)
#   3. Record coverage: how many basins and how much original weight contributed.
#   4. Record weight_at_zero: fraction of buffer weight with score exactly 0.0
#      (used by degenerate-at-floor guard in Cell 7).

distributions = {}   # api_key → {scores, weights, n, coverage_weight, weight_at_zero}

for var in block1_vars:
    col = joined[var].apply(pd.to_numeric, errors='coerce')
    w   = joined['weight']

    mask   = col.notna()
    scores = col[mask].values.astype(float)
    wts    = w[mask].values.astype(float)

    coverage_weight     = wts.sum()
    wts_norm            = wts / coverage_weight
    weight_at_zero_frac = float(wts[scores == 0.0].sum()) if scores.size > 0 else 0.0

    distributions[var] = {
        'scores':          scores,
        'weights':         wts_norm,
        'n':               int(mask.sum()),
        'coverage_weight': round(float(coverage_weight), 4),
        'weight_at_zero':  round(weight_at_zero_frac, 4),
    }

print(f'Distributions assembled for {len(distributions)} variables')
dropped = [(v, d) for v, d in distributions.items() if d['n'] < len(joined)]
if dropped:
    print('Variables with null-dropped basins:')
    for v, d in dropped:
        print(f'  {v}: {d["n"]}/{len(joined)} basins, coverage_weight={d["coverage_weight"]}')
else:
    print('No null-dropped basins in block-1 variables')

waz_hits = [(v, d['weight_at_zero']) for v, d in distributions.items() if d['weight_at_zero'] > 0]
if waz_hits:
    print(f'\nVariables with buffer weight at score=0:')
    for v, waz in sorted(waz_hits, key=lambda x: -x[1]):
        print(f'  {v:35s}  weight_at_zero={waz:.3f}')

Distributions assembled for 34 variables
Variables with null-dropped basins:
  pct_clay: 8/9 basins, coverage_weight=0.8372
  pct_clay_upstream: 8/9 basins, coverage_weight=0.8372
  pct_silt: 8/9 basins, coverage_weight=0.8372
  pct_silt_upstream: 8/9 basins, coverage_weight=0.8372
  pct_sand: 8/9 basins, coverage_weight=0.8372
  pct_sand_upstream: 8/9 basins, coverage_weight=0.8372

Variables with buffer weight at score=0:
  karst                                weight_at_zero=1.000
  karst_upstream                       weight_at_zero=1.000
  permafrost_extent                    weight_at_zero=1.000
  cropland_extent                      weight_at_zero=0.796
  cropland_extent_upstream             weight_at_zero=0.553
  wet_pct_grp1                         weight_at_zero=0.465
  wet_pct_grp1_upstream                weight_at_zero=0.465
  wet_pct_grp2                         weight_at_zero=0.465
  wet_pct_grp2_upstream                weight_at_zero=0.465
  dist_sink                     

In [6]:
# Cell 6 — Step 3.4: coherence statistics
#
# For each variable:
#   weighted_mean = sum(score_i * weight_i)
#   weighted p10, p90 via sorted cumulative weights + linear interpolation
#   spread = p90 - p10  (in percentile points)

def weighted_quantile(scores, weights, q):
    """Weighted quantile via sorted cumulative weights, linear interpolation."""
    sort_idx = np.argsort(scores)
    s = scores[sort_idx]
    w = weights[sort_idx]
    cumw = np.cumsum(w)
    cumw /= cumw[-1]   # ensure sums to 1 after float rounding
    return float(np.interp(q, cumw, s))

stats = {}
for var, d in distributions.items():
    s, w = d['scores'], d['weights']
    wmean = float(np.dot(s, w))
    p10   = weighted_quantile(s, w, P_LOW  / 100)
    p90   = weighted_quantile(s, w, P_HIGH / 100)
    spread = p90 - p10
    stats[var] = {
        'weighted_mean':    round(wmean,  2),
        'p10':              round(p10,    2),
        'p90':              round(p90,    2),
        'spread':           round(spread, 2),
        'n':                d['n'],
        'coverage_weight':  d['coverage_weight'],
    }

stats_df = pd.DataFrame(stats).T.sort_values('spread')
print(f'Coherence statistics — block 1 ({len(stats_df)} variables)')
print(f'Spread threshold T = {SPREAD_THRESHOLD} percentile points')
print()
display(stats_df)

,weighted_mean,p10,p90,spread,n,coverage_weight
karst,0.00,0.00,0.00,0.00,9.0,1.0000
karst_upstream,0.00,0.00,0.00,0.00,9.0,1.0000
permafrost_extent,0.00,0.00,0.00,0.00,9.0,1.0000
temp_yr,97.85,96.29,98.32,2.03,9.0,1.0000
elev_min,63.20,62.02,64.16,2.13,9.0,1.0000
aridity,10.19,6.26,13.40,7.13,9.0,1.0000
temp_yr_upstream,96.45,91.08,99.11,8.03,9.0,1.0000
stream_gradient,5.43,2.38,11.45,9.07,9.0,1.0000
precip_yr,16.11,9.04,22.86,13.83,9.0,1.0000
cropland_extent,5.29,0.00,14.14,14.14,9.0,1.0000


In [7]:
# Cell 7 — Step 3.5: classify and emit block-1 results
#
# Shared output envelope (all blocks):
#   variable, method, status, representative_score, representative_raw,
#   n_basins, coverage_weight
# Block-1 detail: spread, p10, p90, weight_at_zero
# Block-2 detail: dominant_hybas_id  (null here)

results = []
for var, s in stats.items():
    zf = None
    if var in meta_df.index and 'zero_fraction' in meta_df.columns:
        try:
            zf = float(meta_df.loc[var, 'zero_fraction'])
            if np.isnan(zf):
                zf = None
        except (TypeError, ValueError):
            zf = None

    waz = distributions[var].get('weight_at_zero', 0.0)

    if zf is not None and zf >= ZERO_FRACTION_THRESHOLD and waz >= ZERO_COVERAGE_THRESHOLD:
        status    = 'outside_active_domain'
        rep_score = None
    elif s['spread'] < SPREAD_THRESHOLD:
        status    = 'concentrated'
        rep_score = s['weighted_mean']
    else:
        status    = 'spread'
        rep_score = None

    results.append({
        'variable':             var,
        'method':               'area_weighted',
        'status':               status,
        'representative_score': rep_score,
        'representative_raw':   None,          # native-unit means deferred
        'n_basins':             s['n'],
        'coverage_weight':      s['coverage_weight'],
        # block-1 detail
        'spread':               s['spread'],
        'p10':                  s['p10'],
        'p90':                  s['p90'],
        'weight_at_zero':       waz,
        # block-2 detail (null for block 1)
        'dominant_hybas_id':    None,
    })

results_df = pd.DataFrame(results).set_index('variable').sort_values('spread')

outside      = results_df[results_df['status'] == 'outside_active_domain']
concentrated = results_df[results_df['status'] == 'concentrated']
spread_vars  = results_df[results_df['status'] == 'spread']

print(f'Block 1 results — T = {SPREAD_THRESHOLD}')
print(f'  outside_active_domain : {len(outside)}')
print(f'  concentrated          : {len(concentrated)}')
print(f'  spread                : {len(spread_vars)}')
print()
if len(outside):
    print('=== OUTSIDE ACTIVE DOMAIN ===')
    display(outside[['spread', 'p10', 'p90', 'weight_at_zero', 'n_basins']])
    print()
print('=== CONCENTRATED (representative_score reported) ===')
display(concentrated[['representative_score', 'spread', 'p10', 'p90', 'n_basins', 'coverage_weight']])
print()
print('=== SPREAD (no single value) ===')
display(spread_vars[['spread', 'p10', 'p90', 'n_basins', 'coverage_weight', 'weight_at_zero']])

results_df.to_csv(OUT / 'step3_block1_results.tsv', sep='\t', float_format='%.2f')
print(f'\nSaved step3_block1_results.tsv')


Saved step3_block1_results.tsv


In [8]:
# Cell 8 — Block 2: network-topology (discharge_annual, discharge_min)
#
# Discharge is cumulative — each basin already integrates upstream flow.
# No mean, no area-weighted distribution. Report the dominant river only.
# Dominant = basin with highest annual discharge in the buffer set.
# Both variables read from that one basin; discharge_min > 0 → perennial,
# discharge_min = 0 → seasonal / intermittent.

nt_vars      = meta_df[meta_df['typology_cluster'] == 'network-topology']
dominant_id  = int(raw_df['discharge_yr'].idxmax())
n_total      = len(raw_df)

block2_rows = []
for api_key, row in nt_vars.iterrows():
    score   = round(float(matrix_df.loc[dominant_id, api_key]), 2)
    raw_val = round(float(raw_df.loc[dominant_id, api_key]),    3)
    block2_rows.append({
        'variable':             api_key,
        'method':               'dominant_basin',
        'status':               'dominant',
        'representative_score': score,
        'representative_raw':   raw_val,
        'n_basins':             n_total,
        'coverage_weight':      1.0,
        'spread':               np.nan,
        'p10':                  np.nan,
        'p90':                  np.nan,
        'weight_at_zero':       np.nan,
        'dominant_hybas_id':    dominant_id,
    })

block2_df = pd.DataFrame(block2_rows).set_index('variable')

print(f'Block 2 — network-topology  (dominant basin: hybas_id {dominant_id})')
print(f'  annual discharge (m³/yr) : {raw_df.loc[dominant_id, "discharge_yr"]:.1f}')
print(f'  min    discharge (m³/mn) : {raw_df.loc[dominant_id, "discharge_min"]:.1f}')
print()
display(block2_df[['method', 'status', 'representative_score', 'representative_raw', 'dominant_hybas_id']])

# Align dtypes before concat to avoid FutureWarning about all-NA column inference.
# Block 1 has all-None: dominant_hybas_id (→ Int64) and representative_raw (→ float64).
# Block 2 has all-NaN: spread, p10, p90, weight_at_zero (→ float64, already, but explicit).
results_df['dominant_hybas_id']  = results_df['dominant_hybas_id'].astype('Int64')
block2_df['dominant_hybas_id']   = block2_df['dominant_hybas_id'].astype('Int64')
results_df['representative_raw'] = results_df['representative_raw'].astype('float64')
for col in ['spread', 'p10', 'p90', 'weight_at_zero']:
    block2_df[col] = block2_df[col].astype('float64')

combined_df = pd.concat([results_df, block2_df])
combined_df.to_csv(OUT / 'step3_results.tsv', sep='\t', float_format='%.2f')
print(f'\nSaved step3_results.tsv  ({len(combined_df)} variables total)')


Saved step3_results.tsv  (37 variables total)


In [9]:
# Cell 9 — Block 3 setup: constants, variable set, label map
#
# Categoricals dispatch by position_method='rarity_rank'.
# api_key_s names match the view's TEXT column names (human-readable for sandbox/payload),
# but step2_class_ids stores INTEGERS under those same names (fetched from raw table).
# This is intentional; do not confuse the name with the dtype.
#
# Label lookup: query v_basin06_persist_rev1 for buffer basins — the view joins lu_*
# internally and exposes both integer ID and text label columns per variable.
# VIEW_COL_MAP: api_key → (view_integer_col, view_text_col)
#
# Dedup on basin08_col_s: tec_cl_smj appears as both eco_id and ecoregion (same physical
# column); drop_duplicates keeps eco_id (the stable integer key row).
# Excluded:
#   coast_flag, endorheic → Block 4
#   pnv_shares            → deferred (compositional object)
#   strata_code           → opaque sub-zone codes (e.g. "Q5"); zone_name already covers
#                           the human-readable zone group; strata differences within a zone
#                           are undocumented in the source publication

import importlib
importlib.reload(_dbu)   # ensure read_areas_tsv is available if kernel predates the addition

PLURALITY_THRESHOLD = 0.85   # modal share >= this → 'concentrated'; high to preserve heterogeneity
MIN_SHARE_EPSILON   = 1e-6   # drop sliver classes below this weight fraction
LOW_COVERAGE_FLOOR  = 0.50   # status='low_coverage' if coverage_weight below this
VIEW_TABLE          = 'v_basin06_persist_rev1'
EXCLUDE_CATS        = {'coast_flag', 'endorheic', 'pnv_shares', 'strata_code'}

VIEW_COL_MAP = {
    'lith_class':                 ('lithology',          'lith_class'),
    'wetland_class':              ('wetland_class_id',   'wetland_class'),
    'zone_name':                  ('zone_id',            'zone_name'),
    'biome':                      ('biome_id',           'biome'),
    'eco_id':                     ('eco_id',             'ecoregion'),
    'pnv_majority':               ('pnveg_id',           'pnv_majority'),
    'freshwater_ecoregion_class': ('freshwater_type',    'freshwater_ecoregion_class'),
    'freshwater_ecoregion_name':  ('freshwater_ecoreg',  'freshwater_ecoregion_name'),
    'land_cover_name':            ('land_cover_id',      'land_cover_name'),
}

cat   = pd.read_csv(ROOT / 'documentation' / 'EDOPS_variable_catalog_v0.3.tsv', sep='\t')
b3_cat = cat[
    (cat['status'] == 'implemented') &
    (cat['position_method'] == 'rarity_rank') &
    (~cat['api_key_s'].isin(EXCLUDE_CATS)) &
    (cat['api_key_s'].notna())
].drop_duplicates(subset='basin08_col_s')   # tec_cl_smj dedup: keeps eco_id row

b3_vars = b3_cat['api_key_s'].tolist()

assert not [v for v in EXCLUDE_CATS if v in b3_vars], \
    f"Excluded vars found in working set: {[v for v in EXCLUDE_CATS if v in b3_vars]}"

print(f'Block 3 variables ({len(b3_vars)}):')
for _, row in b3_cat.iterrows():
    print(f'  {row["api_key_s"]:35s}  db_col={row["basin08_col_s"]}')

# Query view for text labels + integer IDs on buffer basins
hybas_ids = raw_df.index.tolist()
ids_str   = ','.join(str(h) for h in hybas_ids)
all_view_cols = ['hybas_id'] + list(dict.fromkeys(
    [c for v in b3_vars if v in VIEW_COL_MAP for c in VIEW_COL_MAP[v]]
))
sql  = f"SELECT {', '.join(all_view_cols)} FROM {VIEW_TABLE} WHERE hybas_id IN ({ids_str})"
conn = _dbu.db_connect()
view_df = pd.read_sql(sql, conn)
conn.close()

# Build {api_key: {class_id(int): label(str)}} — NULL text label = NoData, excluded here
label_map = {}
for api_key in b3_vars:
    if api_key not in VIEW_COL_MAP:
        print(f'  WARNING: no VIEW_COL_MAP entry for {api_key}')
        continue
    int_col, text_col = VIEW_COL_MAP[api_key]
    pairs = view_df[[int_col, text_col]].dropna(subset=[text_col]).drop_duplicates()
    label_map[api_key] = dict(zip(pairs[int_col].astype(int), pairs[text_col]))

print(f'\nLabel map built for {len(label_map)} variables:')
for v, m in label_map.items():
    print(f'  {v:35s}  {len(m)} classes: {list(m.values())[:2]}')

/var/folders/f9/r5mr431d23zcpktsjz_xjxt40000gn/T/ipykernel_32807/297586805.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  view_df = pd.read_sql(sql, conn)


In [ ]:
# Cell 10 — load step2_class_ids, join to buffer weights
#
# step2_class_ids: integer class IDs fetched from raw table via db_col.
# Column names are api_key_s (e.g. 'lith_class') but values are integers — by design.
# read_areas_tsv forces hybas_id to Int64.

class_ids_df = _dbu.read_areas_tsv(OUT / 'step2_class_ids.tsv', index_col='hybas_id')
weights_df   = raw_df[['weight']].copy()

b3_joined = weights_df.join(class_ids_df[b3_vars], how='inner')
assert len(b3_joined) == len(weights_df), \
    f'Join size mismatch: {len(b3_joined)} vs {len(weights_df)}'

print(f'b3_joined: {b3_joined.shape[0]} basins × {len(b3_vars)} vars + weight')
print(b3_joined.to_string())

In [ ]:
# Cell 11 — aggregate: weighted class mixture per variable
#
# For each variable:
#   - NoData = class_id not in label_map (NULL text label in view = no class assigned)
#   - Drop NoData basins; record coverage_weight = sum(surviving weights) / total_weight
#   - Renormalize surviving weights to sum to 1 within the surviving set
#   - Group by class_id, sum renormalized weights → proportion per class
#   - Drop classes below MIN_SHARE_EPSILON; attach labels; sort descending

total_weight = b3_joined['weight'].sum()

mixtures  = {}  # api_key → list of {class_id, label, proportion} sorted desc
coverages = {}  # api_key → {n_basins, coverage_weight}

for var in b3_vars:
    col = b3_joined[var].copy()
    w   = b3_joined['weight'].copy()

    valid_ids = set(label_map.get(var, {}).keys())
    mask      = col.isin(valid_ids)

    surviving_w = w[mask]
    cov_weight  = float(surviving_w.sum())

    if cov_weight < 1e-9:
        mixtures[var]  = []
        coverages[var] = {'n_basins': 0, 'coverage_weight': 0.0}
        continue

    w_norm = surviving_w / cov_weight
    ids    = col[mask]

    props = (
        pd.DataFrame({'class_id': ids.values.astype(int), 'w': w_norm.values})
        .groupby('class_id')['w'].sum()
        .reset_index()
        .rename(columns={'w': 'proportion'})
    )
    props = props[props['proportion'] >= MIN_SHARE_EPSILON].copy()
    props['label'] = props['class_id'].map(label_map[var])
    props = props.sort_values('proportion', ascending=False).reset_index(drop=True)

    mixtures[var]  = props.to_dict('records')
    coverages[var] = {'n_basins': int(mask.sum()), 'coverage_weight': round(cov_weight, 4)}

print(f'Mixtures assembled for {len(mixtures)} variables:')
for var in b3_vars:
    cov = coverages[var]
    mix = mixtures[var]
    top = f"{mix[0]['label']} ({mix[0]['proportion']:.3f})" if mix else 'empty'
    print(f'  {var:35s}  n={cov["n_basins"]}  cov={cov["coverage_weight"]:.3f}  modal={top}')

In [ ]:
# Cell 12 — modal class, HHI concentration, verdict
#
# modal_class_id/share/label = argmax of mixture proportions.
# concentration = HHI = Σ(pᵢ²) — continuous signal carried in detail cols.
# verdict = 'concentrated' if modal_share >= PLURALITY_THRESHOLD else 'mixed'.

modals = {}

for var in b3_vars:
    mix = mixtures[var]
    if not mix:
        modals[var] = {
            'modal_class_id': None, 'modal_label': None, 'modal_share': None,
            'n_classes': 0, 'concentration': None, 'verdict': 'no_data',
        }
        continue

    modal     = mix[0]
    n_classes = len(mix)
    hhi       = sum(r['proportion'] ** 2 for r in mix)
    verdict   = 'concentrated' if modal['proportion'] >= PLURALITY_THRESHOLD else 'mixed'

    modals[var] = {
        'modal_class_id':  modal['class_id'],
        'modal_label':     modal['label'],
        'modal_share':     round(modal['proportion'], 4),
        'n_classes':       n_classes,
        'concentration':   round(hhi, 4),
        'verdict':         verdict,
    }

modal_df = pd.DataFrame(modals).T
print('Block 3 verdicts:')
display(modal_df[['modal_label', 'modal_share', 'n_classes', 'concentration', 'verdict']])

In [ ]:
# Cell 13 — assemble output frames in memory (no writes until Cell 15 post-validation)
#
# Headline rows: one per variable → step3_results.tsv (shared envelope).
# Block-3 detail cols added here: modal_class_id, modal_share, n_classes, concentration, verdict.
# Blocks 1+2 rows will receive NaN for these cols when merged in Cell 15.
#
# Companion rows: one per (variable, class) → step3_block3_mixture.tsv.

b3_headline_rows = []
for var in b3_vars:
    cov = coverages[var]
    mod = modals[var]
    if mod['verdict'] == 'no_data':
        status = 'no_data'
    elif cov['coverage_weight'] < LOW_COVERAGE_FLOOR:
        status = 'low_coverage'
    else:
        status = 'ok'

    b3_headline_rows.append({
        'variable':             var,
        'method':               'class_mixture',
        'status':               status,
        'representative_score': np.nan,
        'representative_raw':   mod['modal_label'],
        'n_basins':             cov['n_basins'],
        'coverage_weight':      cov['coverage_weight'],
        # block-1 detail (not applicable)
        'spread':               np.nan,
        'p10':                  np.nan,
        'p90':                  np.nan,
        'weight_at_zero':       np.nan,
        # block-2 detail (not applicable)
        'dominant_hybas_id':    pd.NA,
        # block-3 detail
        'modal_class_id':       mod['modal_class_id'] if mod['modal_class_id'] is not None else pd.NA,
        'modal_share':          mod['modal_share'],
        'n_classes':            mod['n_classes'],
        'concentration':        mod['concentration'],
        'verdict':              mod['verdict'],
    })

b3_headline_df = pd.DataFrame(b3_headline_rows).set_index('variable')
b3_headline_df['dominant_hybas_id'] = b3_headline_df['dominant_hybas_id'].astype('Int64')
b3_headline_df['modal_class_id']    = b3_headline_df['modal_class_id'].astype('Int64')

mixture_rows = []
for var, mix in mixtures.items():
    for r in mix:
        mixture_rows.append({
            'variable':        var,
            'class_id':        r['class_id'],
            'class_label':     r['label'],
            'weight_fraction': round(r['proportion'], 6),
        })
b3_mixture_df = pd.DataFrame(mixture_rows)

print(f'Headline rows : {len(b3_headline_df)}')
print(f'Mixture rows  : {len(b3_mixture_df)}')
print()
display(b3_headline_df[['method', 'status', 'representative_raw', 'modal_share', 'n_classes', 'verdict']])

In [ ]:
# Cell 14 — validation (print only; no writes)
#
# All checks must pass before running Cell 15.

print('=== Block 3 validation ===\n')

# 1. Σ weight_fraction ≈ 1.0 per variable (renormalized within surviving set)
#    coverage_weight is a separate metadata field (fraction of buffer with valid data);
#    it is NOT expected to equal Σ weight_fraction.
print('1. Σ weight_fraction ≈ 1.0 per variable (renormalized within surviving set):')
mix_sums = b3_mixture_df.groupby('variable')['weight_fraction'].sum()
all_ok = True
for var in b3_vars:
    cov  = coverages[var]
    if cov['coverage_weight'] < 1e-9:
        print(f'  {var:35s}  no data — skipped')
        continue
    actual = float(mix_sums.get(var, 0.0))
    ok     = abs(actual - 1.0) < 1e-5
    if not ok:
        all_ok = False
    flag = '✓' if ok else '✗ MISMATCH'
    print(f'  {var:35s}  Σ={actual:.6f}  coverage={cov["coverage_weight"]:.4f}  {flag}')
print(f'  → {"all OK" if all_ok else "PROBLEMS FOUND"}\n')

# 2. No duplicate variable rows in headline
dupes = int(b3_headline_df.index.duplicated().sum())
print(f'2. Duplicate variable rows in headline: {dupes}  ({"OK" if dupes == 0 else "PROBLEM"})\n')

# 3. No NaN class IDs in mixture
nan_ids = int(b3_mixture_df['class_id'].isna().sum())
print(f'3. NaN class_ids in mixture: {nan_ids}  ({"OK" if nan_ids == 0 else "PROBLEM"})\n')

# 4. Excluded vars absent from outputs
print('4. Excluded vars absent from outputs:')
for v in ['coast_flag', 'endorheic', 'pnv_shares', 'strata_code']:
    present = v in b3_headline_df.index
    print(f'   {v}: present={present}  ({"PROBLEM" if present else "OK"})')
print()

# 5. Timbuktu spot checks — directional sanity
print('5. Top-3 mixture per variable (Timbuktu 100km / L06):')
for var in b3_vars:
    rows = b3_mixture_df[b3_mixture_df['variable'] == var].head(3)
    parts = ', '.join(f"{r['class_label']} ({r['weight_fraction']:.2f})" for _, r in rows.iterrows())
    print(f'  {var:35s}  {parts}')

In [ ]:
# Cell 15 — write (run only after reviewing Cell 14 validation output)
#
# Appends Block 3 headline rows to step3_results.tsv.
#   - Adds block-3 detail cols (NaN / pd.NA) to existing blocks 1+2 rows.
#   - Deduplicates on variable index (keep last — new rows win if re-run).
# Writes step3_block3_mixture.tsv (full class mixture, all variables).

existing = _dbu.read_areas_tsv(OUT / 'step3_results.tsv', index_col='variable')

# Add block-3 detail cols to existing rows so concat doesn't create all-NaN object cols
for col in ['modal_class_id', 'modal_share', 'n_classes', 'concentration', 'verdict']:
    if col not in existing.columns:
        existing[col] = np.nan

# Align integer cols before concat
existing['dominant_hybas_id'] = existing['dominant_hybas_id'].astype('Int64')
existing['modal_class_id']    = existing['modal_class_id'].astype('Int64')

combined = pd.concat([existing, b3_headline_df])
combined = combined[~combined.index.duplicated(keep='last')]
combined.to_csv(OUT / 'step3_results.tsv', sep='\t', float_format='%.4f')

b3_mixture_df.to_csv(OUT / 'step3_block3_mixture.tsv', sep='\t', index=False, float_format='%.6f')

print(f'step3_results.tsv        : {len(combined)} rows  ({len(existing)} prev + {len(b3_headline_df)} new)')
print(f'step3_block3_mixture.tsv : {len(b3_mixture_df)} rows')

In [ ]:
# Cell 16 — Block 4: load flag values and derive outlet_type
#
# endo (endorheic) and coast_flag are emitted as raw integers in step2_raw.tsv.
# outlet_type class_id = endo * 10 + coast_flag  (deterministic integer composite):
#   0  = Exorheic, non-coastal       (endo=0, coast=0)
#   1  = Exorheic, coastal           (endo=0, coast=1)
#  10  = Endorheic (inland sink)     (endo=1, coast=0)
#  20  = Terminal sink basin         (endo=2, coast=0)
# Assert: no basin has coast==1 & endo>=1 (globally impossible per cross-tab).

OUTLET_TYPE_LABELS = {
     0: 'Exorheic, non-coastal',
     1: 'Exorheic, coastal',
    10: 'Endorheic (drains to inland sink)',
    20: 'Terminal sink basin',
}

flags = raw_df[['weight', 'endorheic', 'coast_flag']].copy()
flags['endo']  = flags['endorheic'].astype(int)
flags['coast'] = flags['coast_flag'].astype(int)

# Exclusivity assertion
bad = flags[(flags['coast'] == 1) & (flags['endo'] >= 1)]
assert len(bad) == 0, f'Exclusivity violated: {len(bad)} basin(s) with coast=1 & endo>=1: {bad.index.tolist()}'
print('Exclusivity assertion passed: no basin has coast=1 & endo>=1')

flags['outlet_type_id'] = flags['endo'] * 10 + flags['coast']
flags['outlet_type_label'] = flags['outlet_type_id'].map(OUTLET_TYPE_LABELS)

unknown = flags[flags['outlet_type_label'].isna()]
assert len(unknown) == 0, f'Unknown outlet_type combo(s): {unknown[["endo","coast"]].drop_duplicates().to_dict("records")}'

print(f'Buffer basins: {len(flags)}')
print()
print(flags[['weight', 'endo', 'coast', 'outlet_type_id', 'outlet_type_label']].to_string())


In [ ]:
# Cell 17 — Block 4: aggregate outlet_type (class_mixture) + coast_fraction (flag_fraction)
#
# outlet_type: same class_mixture machinery as Block 3.
#   NoData = outlet_type_label is null (none expected here; assertion already passed).
#   Renormalize surviving weights; compute modal/HHI/verdict.
#
# coast_fraction: area-weighted mean of coast_flag over all basins.
#   Straightforward — no NoData expected (coast_flag is always 0 or 1).

# --- outlet_type class_mixture ---
valid_ids_ot = set(OUTLET_TYPE_LABELS.keys())
mask_ot      = flags['outlet_type_id'].isin(valid_ids_ot)
surv_w_ot    = flags.loc[mask_ot, 'weight']
cov_w_ot     = float(surv_w_ot.sum())

if cov_w_ot > 1e-9:
    w_norm_ot = surv_w_ot / cov_w_ot
    ot_props  = (
        pd.DataFrame({'class_id': flags.loc[mask_ot, 'outlet_type_id'].values,
                      'w':        w_norm_ot.values})
        .groupby('class_id')['w'].sum()
        .reset_index()
        .rename(columns={'w': 'proportion'})
    )
    ot_props = ot_props[ot_props['proportion'] >= MIN_SHARE_EPSILON].copy()
    ot_props['label'] = ot_props['class_id'].map(OUTLET_TYPE_LABELS)
    ot_props = ot_props.sort_values('proportion', ascending=False).reset_index(drop=True)
else:
    ot_props = pd.DataFrame(columns=['class_id', 'proportion', 'label'])

ot_n_basins    = int(mask_ot.sum())
ot_modal       = ot_props.iloc[0] if len(ot_props) > 0 else None
ot_n_classes   = len(ot_props)
ot_hhi         = float((ot_props['proportion'] ** 2).sum()) if len(ot_props) > 0 else None
ot_modal_share = round(float(ot_modal['proportion']), 4)  if ot_modal is not None else None
ot_verdict     = 'concentrated' if (ot_modal_share is not None and ot_modal_share >= PLURALITY_THRESHOLD) else 'mixed'

print('outlet_type mixture:')
print(ot_props.to_string(index=False))
print(f'  modal: {ot_modal["label"]} ({ot_modal_share:.4f})')
print(f'  n_classes={ot_n_classes}  HHI={ot_hhi:.4f}  verdict={ot_verdict}')
print()

# --- coast_fraction ---
coast_cov_w    = float(flags['weight'].sum())   # all basins have coast_flag; no NoData
coast_fraction = float((flags['coast'] * flags['weight']).sum())   # weighted mean
coast_n        = len(flags)
coast_status   = 'uniform' if coast_fraction in (0.0, 1.0) else 'ok'

print(f'coast_fraction: {coast_fraction:.6f}  n_basins={coast_n}  status={coast_status}')


In [ ]:
# Cell 18 — assemble Block 4 output frames in memory (no writes)
#
# outlet_type → one class_mixture headline row + mixture rows
# coast_fraction → one flag_fraction headline row

ot_status = 'ok' if cov_w_ot >= LOW_COVERAGE_FLOOR else 'low_coverage'

b4_headline_rows = [
    {
        'variable':             'outlet_type',
        'method':               'class_mixture',
        'status':               ot_status,
        'representative_score': np.nan,
        'representative_raw':   ot_modal['label'] if ot_modal is not None else None,
        'n_basins':             ot_n_basins,
        'coverage_weight':      round(cov_w_ot, 4),
        # block-1 detail
        'spread':               np.nan,
        'p10':                  np.nan,
        'p90':                  np.nan,
        'weight_at_zero':       np.nan,
        # block-2 detail
        'dominant_hybas_id':    pd.NA,
        # block-3 detail (reused for block-4 class_mixture)
        'modal_class_id':       int(ot_modal['class_id']) if ot_modal is not None else pd.NA,
        'modal_share':          ot_modal_share,
        'n_classes':            ot_n_classes,
        'concentration':        round(ot_hhi, 4) if ot_hhi is not None else np.nan,
        'verdict':              ot_verdict,
    },
    {
        'variable':             'coast_fraction',
        'method':               'flag_fraction',
        'status':               coast_status,
        'representative_score': np.nan,
        'representative_raw':   round(coast_fraction, 6),
        'n_basins':             coast_n,
        'coverage_weight':      round(coast_cov_w, 4),
        # all detail cols null
        'spread':               np.nan,
        'p10':                  np.nan,
        'p90':                  np.nan,
        'weight_at_zero':       np.nan,
        'dominant_hybas_id':    pd.NA,
        'modal_class_id':       pd.NA,
        'modal_share':          np.nan,
        'n_classes':            np.nan,
        'concentration':        np.nan,
        'verdict':              np.nan,
    },
]

b4_headline_df = pd.DataFrame(b4_headline_rows).set_index('variable')
b4_headline_df['dominant_hybas_id'] = b4_headline_df['dominant_hybas_id'].astype('Int64')
b4_headline_df['modal_class_id']    = b4_headline_df['modal_class_id'].astype('Int64')

b4_mixture_rows = [
    {
        'variable':        'outlet_type',
        'class_id':        int(r['class_id']),
        'class_label':     r['label'],
        'weight_fraction': round(float(r['proportion']), 6),
    }
    for _, r in ot_props.iterrows()
]
b4_mixture_df = pd.DataFrame(b4_mixture_rows)

print('Block 4 headline:')
display(b4_headline_df[['method', 'status', 'representative_raw', 'modal_share', 'n_classes', 'verdict']])
print()
print('Block 4 mixture:')
display(b4_mixture_df)


In [ ]:
# Cell 19 — validation (print only; no writes)

print('=== Block 4 validation ===\n')

# 1. Mixture sums to 1.0
mix_sum = b4_mixture_df['weight_fraction'].sum()
ok1 = abs(mix_sum - 1.0) < 1e-5
print(f'1. outlet_type Σ weight_fraction = {mix_sum:.6f}  ({"OK" if ok1 else "PROBLEM"})')

# 2. Int64 class IDs in mixture (no floats)
ok2 = all(isinstance(v, (int, np.integer)) for v in b4_mixture_df['class_id'])
print(f'2. class_id types are int : {ok2}  ({"OK" if ok2 else "PROBLEM"})')

# 3. Timbuktu: coast_fraction must be 0.0 (deep interior, no coastal basins)
ok3 = coast_fraction == 0.0
print(f'3. coast_fraction == 0.0  : {coast_fraction}  ({"OK" if ok3 else "UNEXPECTED — check fixture"})')

# 4. No 'Exorheic, coastal' class in mixture (follows from coast_fraction == 0)
coastal_class = b4_mixture_df[b4_mixture_df['class_label'] == 'Exorheic, coastal']
ok4 = len(coastal_class) == 0
print(f'4. No coastal class in mixture: {ok4}  ({"OK" if ok4 else "PROBLEM"})')

# 5. Cross-check: endorheic fraction (classes 10 + 20) vs Block 1 dist_sink weight_at_zero
endo_fraction = float(b4_mixture_df[b4_mixture_df['class_id'].isin([10, 20])]['weight_fraction'].sum())
dist_sink_waz = distributions.get('dist_sink', {}).get('weight_at_zero', None)
print(f'5. Cross-check (informational, not assertion):')
print(f'   outlet_type endorheic fraction (classes 10+20) : {endo_fraction:.4f}')
print(f'   Block 1  dist_sink  weight_at_zero             : {dist_sink_waz}')
print(f'   Gap notes: dist_sink zero = "at or near terminal sink" (endo=2 dominant);')
print(f'              outlet_type endorheic includes endo=1 (inland drainage) + endo=2 (terminal).')
print(f'              Expect endo_fraction >= weight_at_zero.')

print()
print('All hard assertions passed.' if all([ok1, ok2, ok3, ok4]) else 'SOME ASSERTIONS FAILED — fix before Cell 20.')


In [ ]:
# Cell 20 — write (run only after reviewing Cell 19 validation output)
#
# Appends Block 4 headline rows to step3_results.tsv.
# Appends outlet_type mixture rows to step3_block3_mixture.tsv.

existing = _dbu.read_areas_tsv(OUT / 'step3_results.tsv', index_col='variable')

# Ensure block-3/4 detail cols exist in existing rows
for col in ['modal_class_id', 'modal_share', 'n_classes', 'concentration', 'verdict']:
    if col not in existing.columns:
        existing[col] = np.nan

existing['dominant_hybas_id'] = existing['dominant_hybas_id'].astype('Int64')
existing['modal_class_id']    = existing['modal_class_id'].astype('Int64')

combined = pd.concat([existing, b4_headline_df])
combined = combined[~combined.index.duplicated(keep='last')]
combined.to_csv(OUT / 'step3_results.tsv', sep='\t', float_format='%.4f')

# Append to mixture file
existing_mix = pd.read_csv(OUT / 'step3_block3_mixture.tsv', sep='\t')
# Drop any prior outlet_type rows (idempotent re-run)
existing_mix = existing_mix[existing_mix['variable'] != 'outlet_type']
new_mix = pd.concat([existing_mix, b4_mixture_df], ignore_index=True)
new_mix.to_csv(OUT / 'step3_block3_mixture.tsv', sep='\t', index=False, float_format='%.6f')

print(f'step3_results.tsv        : {len(combined)} rows  ({len(existing)} prev + {len(b4_headline_df)} new)')
print(f'step3_block3_mixture.tsv : {len(new_mix)} rows  ({len(existing_mix)} prev + {len(b4_mixture_df)} new)')


In [11]:
# Cell 21 — Block 5: input selection
#
# Two sub-sets:
#   (a) untyped continuous: kind=='continuous' AND typology_cluster is NaN
#       excludes categoricals and flags, which also have blank typology
#       excludes gdp_avg and human_dev_idx (implementation excluded from signature)
#   (b) river_area: typology_cluster=='local-anomaly' — deliberate exception routed here
#
# Do NOT edit typology_cluster to disambiguate — filter by kind.
# Do NOT hardcode the variable list — derive programmatically.

EXCLUDE_VARS = {'gdp_avg', 'human_dev_idx'}
EXTREME_VARS = ['river_area']   # local-anomaly; hand-added

untyped_mask = (
    (meta_df['kind'] == 'continuous') &
    meta_df['typology_cluster'].isna()
)
untyped_vars = [
    v for v in meta_df[untyped_mask].index
    if v not in EXCLUDE_VARS and v in joined.columns
]

# Assertions
bad_cats  = [v for v in untyped_vars if v in meta_df.index and meta_df.loc[v, 'kind'] == 'categorical']
bad_flags = [v for v in untyped_vars if v in meta_df.index and meta_df.loc[v, 'kind'] == 'flag']
assert not bad_cats,  f'Categorical leaked into untyped set: {bad_cats}'
assert not bad_flags, f'Flag leaked into untyped set: {bad_flags}'
assert 'river_area' not in untyped_vars, \
    'river_area should not be in untyped_vars — it has typology_cluster=local-anomaly'

extreme_present = {v: v in joined.columns for v in EXTREME_VARS}

print(f'Block 5 variable set:')
print(f'  distribution_only (untyped continuous): {len(untyped_vars)} vars')
for v in sorted(untyped_vars):
    band = meta_df.loc[v, 'band'] if v in meta_df.index else '?'
    print(f'    {v:35s}  band={band}')
print(f'\n  extreme (local-anomaly):')
for v in EXTREME_VARS:
    print(f'    {v:35s}  in_matrix={extreme_present[v]}')

all_untyped_cts = meta_df[(meta_df['kind'] == 'continuous') & meta_df['typology_cluster'].isna()]
not_in_matrix = [v for v in all_untyped_cts.index if v not in EXCLUDE_VARS and v not in joined.columns]
if not_in_matrix:
    print(f'\n  (not in step2 matrix, skipped: {not_in_matrix})')
print(f'\nNote: {len(all_untyped_cts) - len(EXCLUDE_VARS & set(all_untyped_cts.index))} untyped-continuous rows in catalog; '
      f'{len(untyped_vars)} in matrix → typing-pass backlog = '
      f'{len(all_untyped_cts) - len(EXCLUDE_VARS & set(all_untyped_cts.index)) - len(untyped_vars)} not yet in step2')

Block 5 variable set:
  distribution_only (untyped continuous): 2 vars
    temp_max                             band=C
    temp_min                             band=C

  extreme (local-anomaly):
    river_area                           in_matrix=True

Note: 2 untyped-continuous rows in catalog; 2 in matrix → typing-pass backlog = 0 not yet in step2


In [12]:
# Cell 22 — Block 5 Method A: distribution_only (untyped continuous)
#
# Per variable: join scores + weights from `joined`, drop NaN, renormalize survivors.
# weighted_mean → representative_score.
# weighted p10/p90/spread (reuses weighted_quantile from Cell 6) — descriptors only;
# no concentrated/spread verdict (the typing presupposition doesn't hold here).
# status = 'untyped' on every row.

b5_dist_rows      = []
b5_companion_rows = []   # full per-basin rows → step3_block5_distribution.tsv

for var in untyped_vars:
    col  = joined[var].apply(pd.to_numeric, errors='coerce')
    w    = joined['weight']
    mask = col.notna()
    if mask.sum() == 0:
        continue

    scores   = col[mask].values.astype(float)
    wts      = w[mask].values.astype(float)
    cov_w    = float(wts.sum())
    wts_norm = wts / cov_w
    n        = int(mask.sum())

    wmean  = float(np.dot(scores, wts_norm))
    p10    = weighted_quantile(scores, wts_norm, P_LOW  / 100)
    p90    = weighted_quantile(scores, wts_norm, P_HIGH / 100)
    spread = p90 - p10

    b5_dist_rows.append({
        'variable':             var,
        'method':               'distribution_only',
        'status':               'untyped',
        'representative_score': round(wmean, 2),
        'representative_raw':   np.nan,
        'n_basins':             n,
        'coverage_weight':      round(cov_w, 4),
        'spread':               round(spread, 2),
        'p10':                  round(p10,    2),
        'p90':                  round(p90,    2),
        'weight_at_zero':       np.nan,
        'dominant_hybas_id':    pd.NA,
        'modal_class_id':       pd.NA,
        'modal_share':          np.nan,
        'n_classes':            np.nan,
        'concentration':        np.nan,
        'verdict':              np.nan,
    })

    # Companion table: full weighted distribution so any quantile is recoverable
    for hid, score, wt in zip(joined.index[mask], scores, w[mask].values):
        b5_companion_rows.append({
            'variable': var,
            'hybas_id': int(hid),
            'weight':   round(float(wt), 6),
            'score':    round(float(score), 4),
        })

b5_dist_df = pd.DataFrame(b5_dist_rows).set_index('variable')
b5_dist_df['dominant_hybas_id'] = b5_dist_df['dominant_hybas_id'].astype('Int64')
b5_dist_df['modal_class_id']    = b5_dist_df['modal_class_id'].astype('Int64')

b5_companion_df = pd.DataFrame(b5_companion_rows)
if len(b5_companion_df):
    b5_companion_df['hybas_id'] = b5_companion_df['hybas_id'].astype('int64')

print(f'distribution_only: {len(b5_dist_df)} vars  ({len(b5_companion_rows)} companion rows)')
print()
print(b5_dist_df[['status', 'representative_score', 'coverage_weight', 'spread', 'p10', 'p90']].to_string())

distribution_only: 2 vars  (18 companion rows)

           status  representative_score  coverage_weight  spread    p10    p90
variable                                                                      
temp_min  untyped                 78.98              1.0    5.65  75.43  81.08
temp_max  untyped                 95.56              1.0    3.23  93.60  96.83


In [13]:
# Cell 23 — Block 5 Method B: extreme (river_area)
#
# Local-anomaly: area-weighting would dilute the spike.
# Select the basin with the highest river_area score (monotone with raw value).
# Report that basin's score and native raw value; carry dominant_hybas_id.

b5_extreme_rows = []

for var in EXTREME_VARS:
    if not extreme_present.get(var, False):
        print(f'SKIP {var}: not in score matrix')
        continue

    col  = joined[var].apply(pd.to_numeric, errors='coerce')
    mask = col.notna()
    cov_w = float(joined.loc[mask, 'weight'].sum())
    n     = int(mask.sum())

    dom_idx   = col.idxmax()
    dom_score = round(float(col.loc[dom_idx]), 2)
    dom_raw   = round(float(raw_df.loc[dom_idx, var]), 3) if var in raw_df.columns else np.nan
    dom_hybas = int(dom_idx)

    b5_extreme_rows.append({
        'variable':             var,
        'method':               'extreme',
        'status':               'ok',
        'representative_score': dom_score,
        'representative_raw':   dom_raw,
        'n_basins':             n,
        'coverage_weight':      round(cov_w, 4),
        'spread':               np.nan,
        'p10':                  np.nan,
        'p90':                  np.nan,
        'weight_at_zero':       np.nan,
        'dominant_hybas_id':    dom_hybas,
        'modal_class_id':       pd.NA,
        'modal_share':          np.nan,
        'n_classes':            np.nan,
        'concentration':        np.nan,
        'verdict':              np.nan,
    })
    print(f'{var}: dominant hybas_id={dom_hybas}  score={dom_score}  raw={dom_raw} km²')

b5_extreme_df = pd.DataFrame(b5_extreme_rows).set_index('variable')
if len(b5_extreme_df):
    b5_extreme_df['dominant_hybas_id'] = b5_extreme_df['dominant_hybas_id'].astype('Int64')
    b5_extreme_df['modal_class_id']    = b5_extreme_df['modal_class_id'].astype('Int64')

river_area: dominant hybas_id=1060582960  score=86.07  raw=4273.403 km²


In [14]:
# Cell 24 — assemble Block 5 in memory (no writes)

b5_df = pd.concat([b5_dist_df, b5_extreme_df] if len(b5_extreme_df) else [b5_dist_df])

print(f'Block 5 total: {len(b5_df)} rows')
print(f'  distribution_only : {(b5_df["method"] == "distribution_only").sum()}')
print(f'  extreme           : {(b5_df["method"] == "extreme").sum()}')
print()
display(b5_df[['method', 'status', 'representative_score', 'representative_raw',
               'n_basins', 'coverage_weight', 'spread', 'p10', 'p90']])

,method,status,representative_score,representative_raw,n_basins,coverage_weight,spread,p10,p90
variable,,,,,,,,,
temp_min,distribution_only,untyped,78.98,NaN,9,1.0,5.65,75.43,81.08
temp_max,distribution_only,untyped,95.56,NaN,9,1.0,3.23,93.60,96.83
river_area,extreme,ok,86.07,4273.403,9,1.0,NaN,NaN,NaN


In [15]:
# Cell 25 — validation (print only; no writes)

print('=== Block 5 validation ===\n')

dist_rows = b5_df[b5_df['method'] == 'distribution_only']

# 1. All distribution_only rows have status='untyped'
ok1 = (dist_rows['status'] == 'untyped').all()
print(f'1. All distribution_only status="untyped" : {ok1}  ({"OK" if ok1 else "PROBLEM"})')

# 2. Weighted quantiles ordered: p10 ≤ mean ≤ p90
ok2 = True
for var, row in dist_rows.iterrows():
    p10, mean, p90 = row['p10'], row['representative_score'], row['p90']
    if not (p10 <= mean <= p90):
        print(f'   FAIL order: {var}  p10={p10}  mean={mean}  p90={p90}')
        ok2 = False
print(f'2. p10 ≤ mean ≤ p90 for all distribution_only : {ok2}  ({"OK" if ok2 else "PROBLEM"})')

# 3. Coverage weights in (0, 1]
ok3 = ((b5_df['coverage_weight'] > 0) & (b5_df['coverage_weight'] <= 1.0)).all()
print(f'3. coverage_weight in (0, 1]                  : {ok3}  ({"OK" if ok3 else "PROBLEM"})')

# 4. No categorical or flag leaked into distribution_only set
bad_kinds = [v for v in dist_rows.index if v in meta_df.index and meta_df.loc[v, 'kind'] in ('categorical', 'flag')]
ok4 = len(bad_kinds) == 0
print(f'4. No categorical/flag in distribution_only   : {ok4}  ({"OK" if ok4 else f"PROBLEM: {bad_kinds}"})')

# 5. river_area dominant basin — cross-check against Block 2
print(f'5. river_area dominant basin:')
if 'river_area' in b5_df.index:
    dom_id = int(b5_df.loc['river_area', 'dominant_hybas_id'])
    b2_dom = int(block2_df['dominant_hybas_id'].iloc[0]) if len(block2_df) else None
    print(f'   river_area dominant hybas_id : {dom_id}')
    if b2_dom:
        match = dom_id == b2_dom
        print(f'   Block 2 dominant (discharge)  : {b2_dom}  → {"same basin" if match else "DIFFERS (note it, do not force)"}')
else:
    print(f'   river_area not in Block 5 (SKIP — not in step2 matrix)')

# 6. Companion table covers exactly the distribution_only variable set
comp_vars = set(b5_companion_df['variable'].unique()) if len(b5_companion_df) else set()
dist_set  = set(dist_rows.index)
ok6 = comp_vars == dist_set
print(f'6. Companion table var set matches distribution_only : {ok6}  ({"OK" if ok6 else f"mismatch: {comp_vars.symmetric_difference(dist_set)}"})')

# Spot-check: 3 variables, directional sanity
print(f'\nSpot-check (first 3 untyped vars by index):')
for var in list(dist_rows.index[:3]):
    row = dist_rows.loc[var]
    print(f'  {var:35s}  mean={row["representative_score"]:.1f}  '
          f'p10={row["p10"]:.1f}  p90={row["p90"]:.1f}  spread={row["spread"]:.1f}  n={int(row["n_basins"])}')

print()
all_ok = all([ok1, ok2, ok3, ok4, ok6])
print('All hard assertions passed.' if all_ok else 'SOME ASSERTIONS FAILED — fix before Cell 26.')

=== Block 5 validation ===

1. All distribution_only status="untyped" : True  (OK)
2. p10 ≤ mean ≤ p90 for all distribution_only : True  (OK)
3. coverage_weight in (0, 1]                  : True  (OK)
4. No categorical/flag in distribution_only   : True  (OK)
5. river_area dominant basin:
   river_area dominant hybas_id : 1060582960
   Block 2 dominant (discharge)  : 1060564960  → DIFFERS (note it, do not force)
6. Companion table var set matches distribution_only : True  (OK)

Spot-check (first 3 untyped vars by index):
  temp_min                             mean=79.0  p10=75.4  p90=81.1  spread=5.7  n=9
  temp_max                             mean=95.6  p10=93.6  p90=96.8  spread=3.2  n=9

All hard assertions passed.


In [16]:
# Cell 26 — write (run only after reviewing Cell 25 validation output)
#
# Appends Block 5 rows to step3_results.tsv.
# Writes step3_block5_distribution.tsv (full weighted per-basin distribution).
#
# Normalises legacy column names from older TSV writes before concat:
#   n_classes_present → n_classes  |  hhi → concentration
# These renames are no-ops when the TSV was written by this notebook run.

existing = _dbu.read_areas_tsv(OUT / 'step3_results.tsv', index_col='variable')

# Normalize any legacy column aliases
col_renames = {'n_classes_present': 'n_classes', 'hhi': 'concentration'}
existing = existing.rename(columns={k: v for k, v in col_renames.items() if k in existing.columns})

# Ensure all block-5 detail cols exist in existing rows (backfill NaN)
for col in ['n_classes', 'concentration', 'verdict']:
    if col not in existing.columns:
        existing[col] = np.nan

existing['dominant_hybas_id'] = existing['dominant_hybas_id'].astype('Int64')
existing['modal_class_id']    = existing['modal_class_id'].astype('Int64')

combined = pd.concat([existing, b5_df])
combined = combined[~combined.index.duplicated(keep='last')]
combined.to_csv(OUT / 'step3_results.tsv', sep='\t', float_format='%.4f')

b5_companion_df.to_csv(OUT / 'step3_block5_distribution.tsv', sep='\t', index=False, float_format='%.4f')

print(f'step3_results.tsv             : {len(combined)} rows  ({len(existing)} prev + {len(b5_df)} new)')
print(f'step3_block5_distribution.tsv : {len(b5_companion_df)} rows')

step3_results.tsv             : 51 rows  (48 prev + 3 new)
step3_block5_distribution.tsv : 18 rows


In [17]:
# Cell 27 — Block 6: modality detection — PROPOSE ONLY, no writes
#
# Detects distribution shape for every distribution-bearing row:
#   method in {area_weighted (Block 1), distribution_only (Block 5)}
# two_regime iff: max internal gap > MODALITY_GAP × spread
#                 AND each side carries ≥ MIN_REGIME_WEIGHT
# Otherwise unimodal, sub-labelled concentrated (spread < T) or broad.
#
# Prints a full change diff — rerun as many times as needed to tune params.
# Detection reads from the in-memory `joined` matrix; reruns don't compound.

MODALITY_GAP      = 0.50   # gap threshold as fraction of spread — provisional
MIN_REGIME_WEIGHT = 0.20   # minimum weight on each side to count as a regime

# ── target rows ───────────────────────────────────────────────────────────────
target_meta = {}
for var, row in results_df.iterrows():
    if row['method'] == 'area_weighted':
        target_meta[var] = {'method': row['method'], 'status': row['status'],
                            'rep_score': row['representative_score'],
                            'spread': row['spread']}
for var, row in b5_dist_df.iterrows():
    if row['method'] == 'distribution_only':
        target_meta[var] = {'method': row['method'], 'status': row['status'],
                            'rep_score': row['representative_score'],
                            'spread': row['spread']}

# ── endorheic basin set for seam cross-check ──────────────────────────────────
endo_hybas = set(raw_df.index[raw_df['endorheic'].astype(int) > 0].astype(int))

# ── detector ──────────────────────────────────────────────────────────────────
def detect_modality(var, spread):
    """Return (modality_label, evidence_dict_or_None, seam_note)."""
    col  = joined[var].apply(pd.to_numeric, errors='coerce')
    w    = joined['weight']
    mask = col.notna()
    if mask.sum() < 2 or spread == 0:
        return 'unimodal', None, None

    scores  = col[mask].values.astype(float)
    weights = w[mask].values.astype(float)
    ids     = joined.index[mask].astype(int).tolist()
    w_norm  = weights / weights.sum()

    idx       = np.argsort(scores)
    s_sorted  = scores[idx]
    w_sorted  = w_norm[idx]
    id_sorted = [ids[i] for i in idx]

    threshold = MODALITY_GAP * spread
    best      = None
    best_gap  = 0.0

    for i in range(len(s_sorted) - 1):
        gap = s_sorted[i+1] - s_sorted[i]
        if gap > threshold:
            lw = float(w_sorted[:i+1].sum())
            rw = float(w_sorted[i+1:].sum())
            if lw >= MIN_REGIME_WEIGHT and rw >= MIN_REGIME_WEIGHT and gap > best_gap:
                best_gap  = gap
                lc = float(np.dot(s_sorted[:i+1], w_sorted[:i+1] / lw))
                rc = float(np.dot(s_sorted[i+1:], w_sorted[i+1:] / rw))
                left_ids  = set(id_sorted[:i+1])
                right_ids = set(id_sorted[i+1:])
                l_endo = left_ids  & endo_hybas
                r_endo = right_ids & endo_hybas
                if len(l_endo) == len(endo_hybas):
                    seam = 'SEAM ALIGNS — all endo basins on left'
                elif len(r_endo) == len(endo_hybas):
                    seam = 'SEAM ALIGNS — all endo basins on right'
                elif max(len(l_endo), len(r_endo)) > 0:
                    dominant = 'left' if len(l_endo) >= len(r_endo) else 'right'
                    seam = f'partial: {max(len(l_endo),len(r_endo))}/{len(endo_hybas)} endo on {dominant}'
                else:
                    seam = 'no endo basins on either side'
                best = {'gap_size':      round(gap, 2),
                        'threshold':     round(threshold, 2),
                        'split_between': (round(s_sorted[i], 2), round(s_sorted[i+1], 2)),
                        'left_weight':   round(lw, 4), 'right_weight': round(rw, 4),
                        'left_center':   round(lc, 2), 'right_center': round(rc, 2),
                        'n_left':        i + 1,        'n_right':      len(s_sorted) - i - 1,
                        'seam':          seam}

    if best:
        return 'two_regime', best, best['seam']

    sub = 'concentrated' if spread < SPREAD_THRESHOLD else 'broad'
    return sub, None, None

# ── print diff ────────────────────────────────────────────────────────────────
print(f'Block 6 modality detection  '
      f'(MODALITY_GAP={MODALITY_GAP}×spread, MIN_REGIME_WEIGHT={MIN_REGIME_WEIGHT})\n')
print(f'{"variable":<32}  {"current_status":<22}  {"rep_score":>9}  proposed')
print('─' * 105)

proposals        = {}
two_regime_count = 0

for var, meta in sorted(target_meta.items()):
    modality, evidence, seam = detect_modality(var, meta['spread'])
    proposals[var] = (modality, evidence)
    cur_status = meta['status']
    score_str  = f'{meta["rep_score"]:7.2f}' if pd.notna(meta['rep_score']) else '   null'

    if modality == 'two_regime':
        two_regime_count += 1
        ev = evidence
        print(f'{var:<32}  {cur_status:<22}  {score_str}  → TWO_REGIME  '
              f'gap={ev["gap_size"]}pp (thr={ev["threshold"]}pp)  '
              f'split=({ev["split_between"][0]}..{ev["split_between"][1]})  '
              f'L:n={ev["n_left"]} w={ev["left_weight"]} ctr={ev["left_center"]}  '
              f'R:n={ev["n_right"]} w={ev["right_weight"]} ctr={ev["right_center"]}')
        print(f'  {"":<32}  {seam}')
    else:
        print(f'{var:<32}  {cur_status:<22}  {score_str}  → {modality}')

print('─' * 105)
print(f'\nSummary: {len(target_meta)} distribution-bearing rows  |  '
      f'two_regime: {two_regime_count}  |  '
      f'unimodal: {len(target_meta) - two_regime_count}')
print('\nNo writes performed. Tune MODALITY_GAP / MIN_REGIME_WEIGHT and rerun if calls look wrong.')

Block 6 modality detection  (MODALITY_GAP=0.5×spread, MIN_REGIME_WEIGHT=0.2)

variable                          current_status          rep_score  proposed
─────────────────────────────────────────────────────────────────────────────────────────────────────────
aridity                           concentrated              10.19  → concentrated
aridity_upstream                  spread                     null  → TWO_REGIME  gap=19.69pp (thr=15.66pp)  split=(16.8..36.48)  L:n=6 w=0.7078 ctr=8.33  R:n=3 w=0.2922 ctr=37.36
                                    SEAM ALIGNS — all endo basins on left
cropland_extent                   concentrated               5.29  → TWO_REGIME  gap=23.56pp (thr=7.07pp)  split=(0.0..23.56)  L:n=6 w=0.7957 ctr=0.0  R:n=3 w=0.2043 ctr=25.91
                                    SEAM ALIGNS — all endo basins on left
cropland_extent_upstream          spread                     null  → TWO_REGIME  gap=32.82pp (thr=22.26pp)  split=(10.03..42.85)  L:n=5 w=0.6897 ctr=1.99

In [20]:
from scripts.shared.db_utils import read_areas_tsv

# Cell 28 — Block 6: write — gated; run ONLY after reviewing Cell 27 diff
#
# Steps:
#   1. Snapshot step3_results.tsv → step3_results.YYYYMMDD.bak  (recovery: copy bak back)
#   2. Add `modality` column to all distribution-bearing rows; null elsewhere
#   3. two_regime rows: null representative_score; stash original in representative_score_suppressed
#   4. Update status = 'two_regime' for two_regime rows
#   5. Validate; write step3_results.tsv + step3_block6_regimes.tsv

import shutil
from datetime import date

RESULTS_PATH = OUT / 'step3_results.tsv'
BAK_PATH     = OUT / f'step3_results.{date.today().strftime("%Y%m%d")}.bak'
REGIMES_PATH = OUT / 'step3_block6_regimes.tsv'

DIST_METHODS = {'area_weighted', 'distribution_only'}

# ── 1. snapshot (written once; skipped on reruns so bak always = pre-run state)
if not BAK_PATH.exists():
    shutil.copy(RESULTS_PATH, BAK_PATH)
    print(f'Snapshot written: {BAK_PATH.name}')
else:
    print(f'Snapshot already exists, skipping: {BAK_PATH.name}')

# ── 2 + 3 + 4. load and apply ─────────────────────────────────────────────────
res = read_areas_tsv(RESULTS_PATH)

if 'modality' not in res.columns:
    res['modality'] = pd.NA
if 'representative_score_suppressed' not in res.columns:
    res['representative_score_suppressed'] = pd.NA

for var, (modality, evidence) in proposals.items():
    mask = res['variable'] == var
    assert mask.sum() == 1, f'Expected 1 row for {var}, got {mask.sum()}'
    res.loc[mask, 'modality'] = modality
    if modality == 'two_regime':
        orig = res.loc[mask, 'representative_score'].values[0]
        if pd.notna(orig):                          # only stash when there was a score to preserve
            res.loc[mask, 'representative_score_suppressed'] = orig
        res.loc[mask, 'representative_score'] = pd.NA
        res.loc[mask, 'status'] = 'two_regime'

# ── 5. validate ───────────────────────────────────────────────────────────────
dist_rows       = res[res['method'].isin(DIST_METHODS)]
two_regime_rows = res[res['modality'] == 'two_regime']
non_dist_rows   = res[~res['method'].isin(DIST_METHODS)]

missing_modality = dist_rows[dist_rows['modality'].isna()]
assert len(missing_modality) == 0, \
    f'Missing modality on distribution rows: {missing_modality["variable"].tolist()}'

two_not_null = two_regime_rows[two_regime_rows['representative_score'].notna()]
assert len(two_not_null) == 0, \
    f'Non-null rep score on two_regime rows: {two_not_null["variable"].tolist()}'

non_dist_with_modality = non_dist_rows[non_dist_rows['modality'].notna()]
assert len(non_dist_with_modality) == 0, \
    f'Unexpected modality on non-distribution rows: {non_dist_with_modality["variable"].tolist()}'

# two_regime rows that had a non-null score before: verify stash is present
had_score = two_regime_rows[two_regime_rows['representative_score_suppressed'].notna()]
print(f'  Stashed pre-existing scores: {had_score["variable"].tolist()}')

print(f'Validation passed:  {len(res)} total rows | '
      f'{len(dist_rows)} distribution | '
      f'{len(two_regime_rows)} two_regime (score suppressed) | '
      f'{len(non_dist_rows)} non-distribution rows unchanged')

# ── 6. build regimes companion ────────────────────────────────────────────────
regime_rows = []
for var, (modality, evidence) in proposals.items():
    if modality != 'two_regime':
        continue
    ev  = evidence
    cov = float(res.loc[res['variable'] == var, 'coverage_weight'].values[0])
    for regime_id, center_key, weight_key, n_key in [
            (0, 'left_center',  'left_weight',  'n_left'),
            (1, 'right_center', 'right_weight', 'n_right')]:
        regime_rows.append({
            'variable':        var,
            'regime_id':       regime_id,
            'regime_center':   ev[center_key],
            'regime_weight':   round(ev[weight_key], 4),
            'n_basins':        ev[n_key],
            'coverage_weight': cov,
        })

regimes_df = pd.DataFrame(regime_rows)

weight_sums = regimes_df.groupby('variable')['regime_weight'].sum()
bad = weight_sums[abs(weight_sums - 1.0) > 1e-4]
assert len(bad) == 0, f'Regime weight sums off: {dict(bad)}'

# ── 7. write ──────────────────────────────────────────────────────────────────
res.to_csv(RESULTS_PATH, sep='\t', index=False)
regimes_df.to_csv(REGIMES_PATH, sep='\t', index=False)

print(f'step3_results.tsv:        {len(res)} rows  '
      f'(modality col added; {len(two_regime_rows)} scores suppressed)')
print(f'step3_block6_regimes.tsv: {len(regimes_df)} rows  '
      f'({len(two_regime_rows)} vars × 2 regimes)')
print(f'Backup:                   {BAK_PATH.name}')

Snapshot written: step3_results.20260619.bak
  Stashed pre-existing scores: ['temp_yr_upstream', 'cropland_extent']
Validation passed:  51 total rows | 36 distribution | 12 two_regime (score suppressed) | 15 non-distribution rows unchanged
step3_results.tsv:        51 rows  (modality col added; 12 scores suppressed)
step3_block6_regimes.tsv: 24 rows  (12 vars × 2 regimes)
Backup:                   step3_results.20260619.bak
